# 📘 PySpark Deep Dive for Data Engineering
## Databricks Notebook — Master the DataFrame API

**What you'll learn:**
- Spark architecture: drivers, executors, partitions, DAGs
- Complete DataFrame API: transformations, actions, joins, windows
- Complex data types: arrays, maps, structs, JSON
- UDFs, performance optimization, writing data
- NEW Healthcare Claims dataset for practice
- 3 mini projects + 5 interview challenges

**Prerequisites:** Notebooks 01 (techmart_dw) and 02 (Python foundations)

---

---
# 🏗️ Setup & Healthcare Dataset Creation

First, verify techmart_dw exists, then create a new Healthcare Claims dataset.

In [ ]:
# Verify techmart_dw
spark.sql("USE techmart_dw")
count = spark.sql("SELECT COUNT(*) FROM orders").collect()[0][0]
print(f"✅ techmart_dw available (orders: {count:,} rows)")

In [ ]:
%sql
-- Create healthcare database
CREATE DATABASE IF NOT EXISTS healthcare_dw;
USE healthcare_dw;

In [ ]:
%sql
-- Providers table (200 rows)
DROP TABLE IF EXISTS providers;
CREATE TABLE providers (
  provider_id INT, name STRING, specialty STRING, hospital STRING,
  npi_number STRING, is_active BOOLEAN
);
INSERT INTO providers
SELECT id AS provider_id,
  CONCAT('Dr. ', CASE abs(hash(id*3))%10 WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams' WHEN 3 THEN 'Brown' WHEN 4 THEN 'Jones' WHEN 5 THEN 'Garcia' WHEN 6 THEN 'Miller' WHEN 7 THEN 'Davis' WHEN 8 THEN 'Rodriguez' ELSE 'Martinez' END) AS name,
  CASE abs(hash(id*7))%8 WHEN 0 THEN 'Cardiology' WHEN 1 THEN 'Orthopedics' WHEN 2 THEN 'Neurology' WHEN 3 THEN 'Oncology' WHEN 4 THEN 'Pediatrics' WHEN 5 THEN 'Emergency' WHEN 6 THEN 'Internal Medicine' ELSE 'Surgery' END AS specialty,
  CASE abs(hash(id*11))%5 WHEN 0 THEN 'City General' WHEN 1 THEN 'St. Mary' WHEN 2 THEN 'University Medical' WHEN 3 THEN 'Regional Health' ELSE 'Community Hospital' END AS hospital,
  CONCAT('NPI', LPAD(CAST(id AS STRING), 8, '0')) AS npi_number,
  CASE WHEN id%12=0 THEN false ELSE true END AS is_active
FROM (SELECT explode(sequence(1, 200)) AS id);

In [ ]:
%sql
-- Patients table (3000 rows)
DROP TABLE IF EXISTS patients;
CREATE TABLE patients (
  patient_id INT, first_name STRING, last_name STRING, dob DATE,
  gender STRING, blood_type STRING, insurance_id STRING,
  primary_physician_id INT, registration_date DATE, contact_json STRING
);
INSERT INTO patients
SELECT id AS patient_id,
  CONCAT('Patient', id) AS first_name,
  CASE abs(hash(id*3))%8 WHEN 0 THEN 'Anderson' WHEN 1 THEN 'Thomas' WHEN 2 THEN 'Jackson' WHEN 3 THEN 'White' WHEN 4 THEN 'Harris' WHEN 5 THEN 'Martin' WHEN 6 THEN 'Thompson' ELSE 'Moore' END AS last_name,
  date_add('1940-01-01', abs(hash(id*7))%25000) AS dob,
  CASE abs(hash(id*11))%3 WHEN 0 THEN 'M' WHEN 1 THEN 'F' ELSE 'Other' END AS gender,
  CASE abs(hash(id*13))%8 WHEN 0 THEN 'A+' WHEN 1 THEN 'A-' WHEN 2 THEN 'B+' WHEN 3 THEN 'B-' WHEN 4 THEN 'AB+' WHEN 5 THEN 'AB-' WHEN 6 THEN 'O+' ELSE 'O-' END AS blood_type,
  CONCAT('INS-', LPAD(CAST(abs(hash(id*17))%50000 AS STRING), 6, '0')) AS insurance_id,
  abs(hash(id*19))%200+1 AS primary_physician_id,
  date_add('2015-01-01', abs(hash(id*23))%3000) AS registration_date,
  CONCAT('{"phone":"+1-555-', LPAD(CAST(abs(hash(id*29))%10000 AS STRING),4,'0'), '","email":"patient', id, '@health.com","emergency_contact":"Contact', abs(hash(id*31))%1000, '"}') AS contact_json
FROM (SELECT explode(sequence(1, 3000)) AS id);

In [ ]:
%sql
-- Diagnoses reference table (500 rows)
DROP TABLE IF EXISTS diagnoses;
CREATE TABLE diagnoses (code STRING, description STRING, category STRING, is_chronic BOOLEAN);
INSERT INTO diagnoses
SELECT
  CONCAT(CASE abs(hash(id*3))%5 WHEN 0 THEN 'A' WHEN 1 THEN 'B' WHEN 2 THEN 'C' WHEN 3 THEN 'D' ELSE 'E' END, LPAD(CAST(id AS STRING),3,'0')) AS code,
  CONCAT('Diagnosis ', id) AS description,
  CASE abs(hash(id*7))%6 WHEN 0 THEN 'Cardiovascular' WHEN 1 THEN 'Respiratory' WHEN 2 THEN 'Musculoskeletal' WHEN 3 THEN 'Neurological' WHEN 4 THEN 'Metabolic' ELSE 'Infectious' END AS category,
  CASE WHEN abs(hash(id*11))%3=0 THEN true ELSE false END AS is_chronic
FROM (SELECT explode(sequence(1, 500)) AS id);

In [ ]:
%sql
-- Claims table (25000 rows) with intentional quality issues
DROP TABLE IF EXISTS claims;
CREATE TABLE claims (
  claim_id INT, patient_id INT, provider_id INT, claim_date DATE,
  diagnosis_code STRING, procedure_code STRING, billed_amount DECIMAL(10,2),
  allowed_amount DECIMAL(10,2), paid_amount DECIMAL(10,2), status STRING,
  claim_type STRING, service_date DATE
);
INSERT INTO claims
SELECT id AS claim_id,
  abs(hash(id*3))%3000+1 AS patient_id,
  abs(hash(id*7))%200+1 AS provider_id,
  date_add('2022-01-01', abs(hash(id*11))%900) AS claim_date,
  CASE WHEN id%15=0 THEN NULL ELSE CONCAT(CASE abs(hash(id*13))%5 WHEN 0 THEN 'A' WHEN 1 THEN 'B' WHEN 2 THEN 'C' WHEN 3 THEN 'D' ELSE 'E' END, LPAD(CAST(abs(hash(id*17))%500+1 AS STRING),3,'0')) END AS diagnosis_code,
  CONCAT('CPT', LPAD(CAST(abs(hash(id*19))%9999+1 AS STRING),5,'0')) AS procedure_code,
  CAST(abs(hash(id*23))%10000+100 AS DECIMAL(10,2)) AS billed_amount,
  CAST(abs(hash(id*23))%10000*0.8+80 AS DECIMAL(10,2)) AS allowed_amount,
  CAST(abs(hash(id*23))%10000*0.7+70 AS DECIMAL(10,2)) AS paid_amount,
  CASE abs(hash(id*29))%5 WHEN 0 THEN 'approved' WHEN 1 THEN 'denied' WHEN 2 THEN 'pending' WHEN 3 THEN 'paid' ELSE 'under_review' END AS status,
  CASE abs(hash(id*31))%3 WHEN 0 THEN 'inpatient' WHEN 1 THEN 'outpatient' ELSE 'emergency' END AS claim_type,
  date_add('2022-01-01', abs(hash(id*11))%900 - abs(hash(id*37))%5) AS service_date
FROM (SELECT explode(sequence(1, 25000)) AS id)
UNION ALL
-- 500 duplicate claims (intentional quality issue)
SELECT id+25000, abs(hash(id*3))%3000+1, abs(hash(id*7))%200+1,
  date_add('2022-01-01', abs(hash(id*11))%900), NULL,
  CONCAT('CPT', LPAD(CAST(abs(hash(id*19))%9999+1 AS STRING),5,'0')),
  CAST(abs(hash(id*23))%10000+100 AS DECIMAL(10,2)),
  CAST(abs(hash(id*23))%10000*0.8+80 AS DECIMAL(10,2)),
  CAST(abs(hash(id*23))%10000*0.7+70 AS DECIMAL(10,2)),
  'pending', 'outpatient',
  date_add('2022-01-01', abs(hash(id*11))%900)
FROM (SELECT explode(sequence(1, 500)) AS id);

In [ ]:
%sql
-- Claim Lines table (75000 rows)
DROP TABLE IF EXISTS claim_lines;
CREATE TABLE claim_lines (
  line_id INT, claim_id INT, procedure_code STRING,
  quantity INT, unit_cost DECIMAL(10,2), line_total DECIMAL(10,2), modifier STRING
);
INSERT INTO claim_lines
SELECT id AS line_id,
  abs(hash(id*3))%25000+1 AS claim_id,
  CONCAT('CPT', LPAD(CAST(abs(hash(id*7))%9999+1 AS STRING),5,'0')) AS procedure_code,
  abs(hash(id*11))%5+1 AS quantity,
  CAST(abs(hash(id*13))%2000+50 AS DECIMAL(10,2)) AS unit_cost,
  CAST((abs(hash(id*11))%5+1)*(abs(hash(id*13))%2000+50) AS DECIMAL(10,2)) AS line_total,
  CASE abs(hash(id*17))%4 WHEN 0 THEN '26' WHEN 1 THEN 'TC' WHEN 2 THEN '59' ELSE NULL END AS modifier
FROM (SELECT explode(sequence(1, 75000)) AS id);

In [ ]:
%sql
-- Verify healthcare tables
SELECT 'providers' AS tbl, COUNT(*) AS rows FROM providers
UNION ALL SELECT 'patients', COUNT(*) FROM patients
UNION ALL SELECT 'diagnoses', COUNT(*) FROM diagnoses
UNION ALL SELECT 'claims', COUNT(*) FROM claims
UNION ALL SELECT 'claim_lines', COUNT(*) FROM claim_lines
ORDER BY tbl;

---
# 📗 PART 0: What is Apache Spark? (Fundamentals)

## Why Spark Exists

Before Spark, big data processing meant **Hadoop MapReduce**:
- Write data to disk after EVERY step
- Extremely slow for iterative algorithms (ML, graph)
- Complex Java code for simple operations

Spark's innovation: **in-memory computing** — keep data in RAM between steps.

```
Hadoop MapReduce:                    Apache Spark:
Step 1 → Write to HDFS              Step 1 → Keep in memory
Read from HDFS → Step 2             Step 2 → Keep in memory
Write to HDFS → Step 3              Step 3 → Keep in memory
Read from HDFS → Result             Result

Time: 30 minutes                    Time: 30 seconds (100x faster!)
```

## Spark vs Pandas vs SQL

| Aspect | Pandas | Spark | SQL (Databricks) |
|--------|--------|-------|-------------------|
| **Data size** | < 10 GB (single machine) | Unlimited (distributed) | Unlimited (distributed) |
| **Execution** | Single core | Multi-node parallel | Multi-node parallel |
| **Language** | Python only | Python, Scala, Java, R, SQL | SQL only |
| **Use case** | Exploration, small data | ETL, large-scale processing | Analytics, BI |
| **Learning curve** | Low | Medium | Low |
| **When to use** | Prototyping, < 10 GB | Production pipelines, > 10 GB | Analyst queries, dashboards |

## Production Scale Examples

| Company | Spark Usage | Scale |
|---------|------------|-------|
| **Netflix** | ETL, recommendations, A/B testing | 100+ PB, 10,000+ jobs/day |
| **Uber** | Real-time pricing, ETA prediction | 1 trillion events/day |
| **Airbnb** | Search ranking, experimentation | 10,000+ Airflow DAGs |
| **Apple** | Siri data processing, analytics | Exabytes of data |
| **Databricks** | The company BUILT on Spark | Powers 10,000+ customers |

> 💡 **Key Insight:** If your data fits in memory on one machine (< 10 GB),
> use Pandas. If it doesn't, use Spark. In production DE, you'll use Spark 90% of the time.

In [ ]:
# What is Spark? — Verify your Spark environment
print("🔥 APACHE SPARK ENVIRONMENT")
print("=" * 60)
print(f"   Spark Version: {spark.version}")
print(f"   Spark App Name: {spark.sparkContext.appName}")
print(f"   Spark Master: {spark.sparkContext.master}")
print(f"   Default Parallelism: {spark.sparkContext.defaultParallelism} cores")
print(f"   Spark UI: Available in Databricks cluster UI")

# Spark components available
print("\n📦 SPARK COMPONENTS:")
print("   ✅ Spark SQL (DataFrames, SQL queries)")
print("   ✅ Spark Streaming (Structured Streaming)")
print("   ✅ MLlib (Machine Learning)")
print("   ✅ GraphX (Graph Processing)")
print("   ✅ Delta Lake (ACID transactions)")

# Quick comparison: Pandas vs Spark
import sys
print(f"\n📊 PANDAS vs SPARK:")
print(f"   Python version: {sys.version.split()[0]}")
try:
    import pandas as pd
    print(f"   Pandas version: {pd.__version__}")
    print(f"   Pandas: Single machine, in-memory, < 10 GB")
except:
    print("   Pandas: not available")
print(f"   Spark: Distributed, disk+memory, unlimited scale")


## Spark Architecture — The Big Picture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        SPARK APPLICATION                                  │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ┌─────────────────────────────────────────────────────────────────┐    │
│  │                      DRIVER PROGRAM                              │    │
│  │                                                                   │    │
│  │  • SparkContext / SparkSession                                   │    │
│  │  • Converts your code into a DAG (execution plan)                │    │
│  │  • Negotiates with Cluster Manager for resources                 │    │
│  │  • Schedules tasks on executors                                  │    │
│  │  • Collects results from executors                               │    │
│  └─────────────────────────────────────────────────────────────────┘    │
│                              │                                           │
│                    ┌─────────┼─────────┐                                │
│                    │         │         │                                │
│  ┌─────────────────▼──┐ ┌───▼─────────────┐ ┌────────────────────┐    │
│  │    EXECUTOR 1       │ │   EXECUTOR 2    │ │    EXECUTOR 3      │    │
│  │                     │ │                 │ │                    │    │
│  │  ┌─────┐ ┌─────┐  │ │ ┌─────┐ ┌─────┐│ │ ┌─────┐ ┌─────┐  │    │
│  │  │Task1│ │Task2│  │ │ │Task3│ │Task4││ │ │Task5│ │Task6│  │    │
│  │  └─────┘ └─────┘  │ │ └─────┘ └─────┘│ │ └─────┘ └─────┘  │    │
│  │                     │ │                 │ │                    │    │
│  │  [Cache/Memory]     │ │ [Cache/Memory]  │ │ [Cache/Memory]    │    │
│  └─────────────────────┘ └─────────────────┘ └────────────────────┘    │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
                              │
                    ┌─────────▼─────────┐
                    │  CLUSTER MANAGER  │
                    │  (Databricks /    │
                    │   YARN / K8s)     │
                    └───────────────────┘
                              │
                    ┌─────────▼─────────┐
                    │  STORAGE (S3 /    │
                    │  ADLS / Delta)    │
                    └───────────────────┘
```

### Key Components

| Component | Role | Analogy |
|-----------|------|---------|
| **Driver** | Orchestrates everything | The conductor of an orchestra |
| **Executor** | Does the actual work | The musicians playing instruments |
| **Cluster Manager** | Allocates resources | The venue manager |
| **Task** | Unit of work on one partition | One musician playing one section |
| **Partition** | Chunk of data | One page of sheet music |
| **Shuffle** | Data redistribution | Musicians swapping seats mid-performance |

In [ ]:
# Demonstrate the RDD vs DataFrame evolution
print("📊 SPARK API EVOLUTION")
print("=" * 60)

# RDD API (Spark 1.x) — low-level, verbose, no optimization
print("\n1️⃣ RDD API (2010-2015) — Low-level, no optimization:")
print("""
   # Old way (DON'T use this anymore):
   rdd = sc.textFile("orders.csv")
   result = (rdd
       .map(lambda line: line.split(","))
       .filter(lambda fields: fields[3] == "completed")
       .map(lambda fields: (fields[1], float(fields[4])))
       .reduceByKey(lambda a, b: a + b))
""")

# DataFrame API (Spark 2.x+) — high-level, optimized by Catalyst
print("2️⃣ DataFrame API (2015+) — High-level, Catalyst-optimized:")
print("""
   # Modern way (USE THIS):
   result = (spark.table("orders")
       .filter(col("status") == "completed")
       .groupBy("customer_id")
       .agg(sum("total_amount").alias("total_spent")))
""")

print("3️⃣ Why DataFrames are better:")
print("   • Catalyst optimizer rewrites your query for performance")
print("   • Tungsten engine generates optimized bytecode")
print("   • Same performance regardless of language (Python = Scala = Java)")
print("   • Schema awareness (catches errors early)")
print("   • Interoperable with SQL")

print("\n💡 Rule: ALWAYS use DataFrames (or Spark SQL). Never use RDDs directly.")
print("   Exception: Very custom partitioning or low-level operations.")


## Transformations vs Actions — Lazy Evaluation

This is the MOST IMPORTANT concept in Spark:

| Type | What It Does | Executes Immediately? | Examples |
|------|-------------|----------------------|----------|
| **Transformation** | Defines a computation | ❌ No (lazy) | filter, select, groupBy, join, withColumn |
| **Action** | Triggers execution | ✅ Yes | show, count, collect, write, save |

### Why Lazy Evaluation?

```python
# Without lazy evaluation (Pandas-style):
df1 = read_data()           # Reads ALL data immediately
df2 = df1.filter(...)       # Processes ALL data immediately
df3 = df2.select(...)       # Processes ALL data immediately
df4 = df3.groupBy(...)      # Processes ALL data immediately
# Each step processes the FULL dataset — wasteful!

# With lazy evaluation (Spark):
df1 = read_data()           # Just records "read data" in the plan
df2 = df1.filter(...)       # Just adds "filter" to the plan
df3 = df2.select(...)       # Just adds "select" to the plan
df4 = df3.groupBy(...)      # Just adds "groupBy" to the plan
df4.show()                  # NOW Spark optimizes and executes everything at once!
# Spark can push the filter down to storage, skip unneeded columns, etc.
```

In [ ]:
# PROOF of lazy evaluation
from pyspark.sql.functions import col, count, sum as spark_sum
import time

print("🔍 PROOF: Lazy Evaluation")
print("=" * 60)

# These are ALL transformations — nothing executes!
start = time.perf_counter()

df1 = spark.table("techmart_dw.orders")
df2 = df1.filter(col("status") == "completed")
df3 = df2.filter(col("total_amount") > 100)
df4 = df3.groupBy("payment_method")
df5 = df4.agg(count("*").alias("cnt"), spark_sum("total_amount").alias("total"))

transform_time = time.perf_counter() - start
print(f"   Transformations defined in: {transform_time:.6f} seconds (instant!)")
print(f"   Nothing has executed yet — Spark just built a plan.")

# THIS is the action — triggers execution
start = time.perf_counter()
result = df5.collect()  # ACTION! Now Spark executes everything
action_time = time.perf_counter() - start
print(f"\n   Action (.collect()) executed in: {action_time:.4f} seconds")
print(f"   THIS is when Spark actually reads data, filters, and aggregates.")

print(f"\n   Result: {len(result)} payment methods")
for row in result[:5]:
    print(f"      {row['payment_method']}: {row['cnt']} orders, ${row['total']:,.2f}")

print("\n💡 Key Takeaway:")
print("   Transformations are FREE (just build a plan).")
print("   Actions are EXPENSIVE (actually process data).")
print("   Spark optimizes the ENTIRE plan before executing.")


In [ ]:
# Narrow vs Wide transformations
print("📊 NARROW vs WIDE TRANSFORMATIONS")
print("=" * 60)

print("""
NARROW (no shuffle — fast):              WIDE (shuffle — expensive):
┌────────┐    ┌────────┐                ┌────────┐    ┌────────┐
│Part. 1 │───▶│Part. 1 │                │Part. 1 │──┐ │Part. A │
└────────┘    └────────┘                └────────┘  │ └────────┘
┌────────┐    ┌────────┐                ┌────────┐  ├▶┌────────┐
│Part. 2 │───▶│Part. 2 │                │Part. 2 │──┤ │Part. B │
└────────┘    └────────┘                └────────┘  │ └────────┘
┌────────┐    ┌────────┐                ┌────────┐  │ ┌────────┐
│Part. 3 │───▶│Part. 3 │                │Part. 3 │──┘ │Part. C │
└────────┘    └────────┘                └────────┘    └────────┘

Each partition processed                 Data MOVES between partitions
independently (no network)               (network I/O = expensive!)
""")

narrow_ops = ["filter()", "select()", "withColumn()", "map()", "flatMap()", "coalesce()"]
wide_ops = ["groupBy()", "join()", "orderBy()", "distinct()", "repartition()", "reduceByKey()"]

print("   NARROW (no shuffle):", ", ".join(narrow_ops))
print("   WIDE (causes shuffle):", ", ".join(wide_ops))

print("\n💡 Performance Rule:")
print("   • Minimize shuffles (wide transformations)")
print("   • Filter BEFORE groupBy/join (reduce data before shuffle)")
print("   • Use broadcast join for small tables (avoids shuffle)")
print("   • Check explain() to see if shuffles are happening")


In [ ]:
# ============================================
# 🎯 YOUR TURN: Identify Transformations vs Actions
# ============================================
# For each operation below, identify:
# 1. Is it a Transformation or Action?
# 2. Is it Narrow or Wide?
# 3. Does it cause a shuffle?
#
# Operations:
# a) df.filter(col("amount") > 100)
# b) df.groupBy("category").count()
# c) df.select("name", "amount")
# d) df.orderBy("amount")
# e) df.show(10)
# f) df.withColumn("tax", col("amount") * 0.08)
# g) df.join(other_df, "id")
# h) df.count()
# i) df.distinct()
# j) df.coalesce(4)
#
# Write your answers below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
answers = [
    ("a) filter(col('amount') > 100)", "Transformation", "Narrow", "No shuffle"),
    ("b) groupBy('category').count()", "Transformation+Action", "Wide", "YES shuffle (groupBy)"),
    ("c) select('name', 'amount')", "Transformation", "Narrow", "No shuffle"),
    ("d) orderBy('amount')", "Transformation", "Wide", "YES shuffle (global sort)"),
    ("e) show(10)", "ACTION", "N/A", "Triggers execution"),
    ("f) withColumn('tax', ...)", "Transformation", "Narrow", "No shuffle"),
    ("g) join(other_df, 'id')", "Transformation", "Wide", "YES shuffle (unless broadcast)"),
    ("h) count()", "ACTION", "Wide", "Triggers execution + shuffle"),
    ("i) distinct()", "Transformation", "Wide", "YES shuffle (compare all rows)"),
    ("j) coalesce(4)", "Transformation", "Narrow", "No shuffle (just combines)"),
]

print("✅ ANSWERS: Transformations vs Actions")
print("=" * 70)
print(f"{'Operation':<35} {'Type':<20} {'Width':<8} {'Shuffle?'}")
print("-" * 70)
for op, op_type, width, shuffle in answers:
    print(f"{op:<35} {op_type:<20} {width:<8} {shuffle}")


---
# 📗 Section 1: Spark Architecture

## Driver vs Executors

```
┌─────────────────────────────────────────────────┐
│                 SPARK APPLICATION                 │
├─────────────────────────────────────────────────┤
│  ┌─────────────┐                                │
│  │   DRIVER    │  - Plans execution (DAG)       │
│  │  (1 node)   │  - Distributes tasks           │
│  │             │  - Collects results             │
│  └──────┬──────┘                                │
│         │                                        │
│    ┌────┴────┬────────┬────────┐                │
│    ▼         ▼        ▼        ▼                │
│ ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐           │
│ │Exec 1│ │Exec 2│ │Exec 3│ │Exec 4│           │
│ │Part 1│ │Part 2│ │Part 3│ │Part 4│           │
│ └──────┘ └──────┘ └──────┘ └──────┘           │
│  EXECUTORS: Process data partitions in parallel  │
└─────────────────────────────────────────────────┘
```

**Key concepts:**
- **Driver:** Brain of the application. Plans, coordinates, collects.
- **Executors:** Workers. Process data partitions in parallel.
- **Partitions:** Chunks of data distributed across executors.
- **DAG:** Directed Acyclic Graph — Spark's execution plan.

## Lazy Evaluation — Transformations vs Actions

**WHY:** Spark doesn't execute anything until you call an ACTION.
This lets it optimize the entire pipeline before running.

**Transformations** (lazy): select, filter, join, groupBy → return a new DataFrame
**Actions** (eager): show, count, collect, write → trigger execution

In [ ]:
# PROOF of lazy evaluation
spark.sql("USE techmart_dw")

# These are TRANSFORMATIONS — nothing executes yet
df = spark.table("orders")
filtered = df.filter("status = 'completed'")
selected = filtered.select("order_id", "total_amount")
sorted_df = selected.orderBy("total_amount", ascending=False)

print("No execution yet! Just building the plan.")
print(f"Type: {type(sorted_df)}")

# THIS is the ACTION — triggers execution
print(f"
Now executing (count): {sorted_df.count()}")

## Narrow vs Wide Transformations

**Narrow** (no shuffle): select, filter, withColumn, union
- Data stays on same partition → FAST

**Wide** (shuffle required): groupBy, join, distinct, orderBy, repartition
- Data moves between executors → EXPENSIVE

In [ ]:
# Demonstrate with explain()
df = spark.table("orders")

# Narrow transformation — no Exchange (shuffle) in plan
print("=== NARROW (filter + select) ===")
narrow = df.filter("status = 'completed'").select("order_id", "total_amount")
narrow.explain(True)

In [ ]:
# Wide transformation — Exchange (shuffle) appears in plan
print("=== WIDE (groupBy) ===")
wide = df.groupBy("status").count()
wide.explain(True)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Identify Shuffles
# ============================================
# For each operation below, predict: NARROW or WIDE?
# Then verify with .explain()
#
# 1. df.filter("total_amount > 1000")
# 2. df.groupBy("payment_method").sum("total_amount")
# 3. df.withColumn("year", year("order_date"))
# 4. df.distinct()
# 5. df.join(spark.table("customers"), "customer_id")
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
df = spark.table("orders")
from pyspark.sql.functions import year

# 1. NARROW — no shuffle
print("1. filter → NARROW")
df.filter("total_amount > 1000").explain()

# 2. WIDE — groupBy causes shuffle
print("
2. groupBy → WIDE")
df.groupBy("payment_method").sum("total_amount").explain()

# 3. NARROW — column operation
print("
3. withColumn → NARROW")
df.withColumn("year", year("order_date")).explain()

# 4. WIDE — distinct requires shuffle
print("
4. distinct → WIDE")
df.select("status").distinct().explain()

# 5. WIDE — join causes shuffle (unless broadcast)
print("
5. join → WIDE")
df.join(spark.table("customers"), "customer_id").explain()

## Spark Execution Model — How Your Code Actually Runs

Understanding the execution model helps you write faster code:

```
YOUR CODE (Python/Scala)
    │
    ▼
DRIVER PROGRAM
    │ Creates a logical plan (DAG of transformations)
    │ Catalyst optimizer rewrites the plan
    │ Tungsten generates optimized bytecode
    ▼
CLUSTER MANAGER (YARN/K8s/Standalone)
    │ Allocates executors
    ▼
EXECUTORS (worker JVMs)
    │ Each executor runs tasks in parallel
    │ Tasks process partitions of data
    ▼
RESULTS → back to driver (for actions like .show(), .collect())
```

### Key Concepts

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Job** | Triggered by an action (show, count, write) | One action = one job |
| **Stage** | Group of tasks separated by shuffles | Shuffles are expensive |
| **Task** | Unit of work on one partition | More partitions = more parallelism |
| **Partition** | Chunk of data processed by one task | Controls parallelism |
| **Shuffle** | Data redistribution across executors | Most expensive operation |

### The Catalyst Optimizer

Spark doesn't execute your code literally — it OPTIMIZES it first:

```python
# You write this:
df.filter(col("status") == "active").select("name", "amount").filter(col("amount") > 100)

# Catalyst rewrites to this (combines filters, pushes down):
df.select("name", "amount", "status").filter((col("status") == "active") & (col("amount") > 100))

# Even better with predicate pushdown to storage:
# Only reads files/partitions where status="active" AND amount>100
```

In [ ]:
# Inspect Spark execution details
print("🔍 SPARK CLUSTER INFORMATION")
print("=" * 60)
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   Default Parallelism: {spark.sparkContext.defaultParallelism}")

# Check key configurations
configs_to_check = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.default.parallelism",
]

print("\n⚙️ Key Configurations:")
for config in configs_to_check:
    try:
        value = spark.conf.get(config)
        print(f"   {config} = {value}")
    except:
        print(f"   {config} = (not set)")


In [ ]:
# Demonstrate lazy evaluation with explain()
from pyspark.sql.functions import col, sum as spark_sum, count, avg

# Build a complex query (nothing executes yet — it's lazy!)
lazy_query = (
    spark.table("techmart_dw.orders")
    .filter(col("status") == "completed")
    .filter(col("total_amount") > 50)
    .groupBy("payment_method")
    .agg(
        count("*").alias("order_count"),
        spark_sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

print("📋 QUERY PLAN (before execution)")
print("=" * 60)
print("Physical Plan:")
lazy_query.explain(mode="simple")

print("\n💡 Notice:")
print("   • Filters are COMBINED and PUSHED DOWN")
print("   • Spark reads only needed columns (column pruning)")
print("   • The plan shows the OPTIMIZED execution strategy")
print("   • Nothing has executed yet — this is just the PLAN")


In [ ]:
# NOW trigger execution with an action
print("\n📊 EXECUTION RESULT:")
lazy_query.show()

print("\n💡 The action (.show()) triggered:")
print("   1. Catalyst optimized the logical plan")
print("   2. Tungsten generated efficient code")
print("   3. Tasks were distributed to executors")
print("   4. Results collected back to driver")

## Partitions — The Key to Parallelism

Data in Spark is divided into **partitions**. Each partition is processed by one task.

```
DataFrame (1 million rows, 8 partitions):
┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐
│Partition 1│ │Partition 2│ │Partition 3│ │Partition 4│
│ 125K rows │ │ 125K rows │ │ 125K rows │ │ 125K rows │
└──────────┘ └──────────┘ └──────────┘ └──────────┘
┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐
│Partition 5│ │Partition 6│ │Partition 7│ │Partition 8│
│ 125K rows │ │ 125K rows │ │ 125K rows │ │ 125K rows │
└──────────┘ └──────────┘ └──────────┘ └──────────┘

8 partitions → 8 tasks → 8 cores working in parallel
```

### Rules of Thumb

| Scenario | Recommended Partitions |
|----------|----------------------|
| Reading files | 1 partition per 128MB file |
| After shuffle | 2-4x number of cores |
| Small data (<1GB) | 8-20 partitions |
| Large data (>100GB) | 200-2000 partitions |
| Writing output | Match target file count |

In [ ]:
# Inspect and control partitions
orders_df = spark.table("techmart_dw.orders")

print("📊 PARTITION ANALYSIS")
print("=" * 60)
print(f"   Current partitions: {orders_df.rdd.getNumPartitions()}")
print(f"   Row count: {orders_df.count():,}")
print(f"   Approx rows per partition: {orders_df.count() // max(orders_df.rdd.getNumPartitions(), 1):,}")

# Repartition (increase parallelism — causes shuffle!)
repartitioned = orders_df.repartition(8)
print(f"\n   After repartition(8): {repartitioned.rdd.getNumPartitions()} partitions")

# Coalesce (decrease partitions — NO shuffle, just combines)
coalesced = orders_df.coalesce(2)
print(f"   After coalesce(2): {coalesced.rdd.getNumPartitions()} partitions")

print("\n💡 Key Difference:")
print("   repartition(N): Shuffles data → even distribution, expensive")
print("   coalesce(N): Combines partitions → no shuffle, but may be uneven")
print("   Rule: Use coalesce to REDUCE, repartition to INCREASE or rebalance")

## The Catalyst Optimizer & Tungsten Engine

### Catalyst: How Spark Optimizes Your Queries

```
YOUR CODE                    CATALYST OPTIMIZER                    EXECUTION
┌──────────┐    ┌──────────────────────────────────────┐    ┌──────────┐
│ DataFrame│───▶│ 1. Parse (build logical plan)        │───▶│ Optimized│
│ or SQL   │    │ 2. Analyze (resolve columns/types)   │    │ Physical │
│ query    │    │ 3. Optimize (rewrite rules)          │    │ Plan     │
│          │    │ 4. Plan (choose join strategy, etc.) │    │          │
└──────────┘    └──────────────────────────────────────┘    └──────────┘
```

### Key Optimizations Catalyst Performs

| Optimization | What It Does | Example |
|-------------|-------------|---------|
| **Predicate Pushdown** | Pushes filters to storage layer | Read only matching files |
| **Column Pruning** | Reads only needed columns | Skip unused columns in Parquet |
| **Constant Folding** | Pre-computes constant expressions | `1 + 1` → `2` at compile time |
| **Filter Reordering** | Puts cheapest filters first | Check boolean before string match |
| **Join Reordering** | Optimizes join order | Smallest table first |
| **Broadcast Detection** | Auto-broadcasts small tables | < 10MB → broadcast join |

### Tungsten: The Execution Engine

Tungsten generates optimized bytecode for your queries:
- **Off-heap memory**: Avoids JVM garbage collection
- **Cache-aware computation**: Optimizes for CPU cache lines
- **Whole-stage code generation**: Fuses operators into single function
- **Vectorized execution**: Processes batches of rows (not one at a time)

In [ ]:
# See Catalyst in action with explain()
from pyspark.sql.functions import col, sum as spark_sum

# Build a query
query = (
    spark.table("techmart_dw.orders")
    .filter(col("status") == "completed")
    .filter(col("total_amount") > 100)
    .filter(col("order_date") >= "2023-01-01")
    .select("customer_id", "total_amount", "order_date")
    .groupBy("customer_id")
    .agg(spark_sum("total_amount").alias("total_spent"))
    .filter(col("total_spent") > 1000)
)

print("📋 CATALYST OPTIMIZER — Query Plan Analysis")
print("=" * 60)
print("\n🔍 Optimized Physical Plan:")
query.explain(mode="simple")

print("\n💡 What Catalyst did:")
print("   1. COMBINED all three filters into one (filter pushdown)")
print("   2. PUSHED filters to storage (reads fewer files)")
print("   3. PRUNED columns (only reads customer_id, total_amount, order_date, status)")
print("   4. CHOSE hash aggregate (efficient for groupBy)")
print("   5. Applied HAVING filter (total_spent > 1000) after aggregation")

In [ ]:
# Compare: explain(mode="formatted") shows more detail
print("📋 DETAILED PLAN (formatted):")
query.explain(mode="formatted")

---
# 📗 Section 2: Reading Data

## Multiple Ways to Read Tables

In [ ]:
# Method 1: spark.table() — simplest for managed tables
orders_df = spark.table("techmart_dw.orders")
print(f"spark.table(): {orders_df.count():,} rows, {len(orders_df.columns)} columns")

# Method 2: spark.read.format("delta")
orders_df2 = spark.read.format("delta").table("techmart_dw.orders")
print(f"spark.read: {orders_df2.count():,} rows")

# Method 3: spark.sql()
orders_df3 = spark.sql("SELECT * FROM techmart_dw.orders")
print(f"spark.sql: {orders_df3.count():,} rows")

## Schema Inference vs Enforcement

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType, TimestampType, BooleanType

# View inferred schema
df = spark.table("techmart_dw.orders")
print("Inferred schema:")
df.printSchema()

---
# 📗 Section 3: Schema Definition

**WHY:** Explicit schemas prevent silent type errors, are faster than inference,
and serve as documentation.

In [ ]:
from pyspark.sql.types import *

# Define schema explicitly
claims_schema = StructType([
    StructField("claim_id", IntegerType(), False),
    StructField("patient_id", IntegerType(), False),
    StructField("provider_id", IntegerType(), True),
    StructField("claim_date", DateType(), True),
    StructField("diagnosis_code", StringType(), True),
    StructField("procedure_code", StringType(), True),
    StructField("billed_amount", DecimalType(10, 2), True),
    StructField("allowed_amount", DecimalType(10, 2), True),
    StructField("paid_amount", DecimalType(10, 2), True),
    StructField("status", StringType(), True),
    StructField("claim_type", StringType(), True),
    StructField("service_date", DateType(), True),
])

print("Defined schema:")
for field in claims_schema.fields:
    nullable = "nullable" if field.nullable else "NOT NULL"
    print(f"  {field.name:<20} {str(field.dataType):<20} {nullable}")

In [ ]:
# DDL string schema (compact alternative)
ddl_schema = "claim_id INT, patient_id INT, provider_id INT, claim_date DATE, diagnosis_code STRING, billed_amount DECIMAL(10,2), status STRING"
print(f"DDL schema: {ddl_schema}")

# Nested schema for JSON
contact_schema = StructType([
    StructField("phone", StringType(), True),
    StructField("email", StringType(), True),
    StructField("emergency_contact", StringType(), True)
])
print(f"\nContact schema fields: {[f.name for f in contact_schema.fields]}")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Define Schema
# ============================================
# Define a StructType schema for a "medical_records" table with:
# - record_id (INT, not null)
# - patient_id (INT, not null)
# - visit_date (DATE)
# - vitals (STRUCT with: blood_pressure STRING, heart_rate INT, temperature DECIMAL(4,1))
# - notes (STRING)
# - medications (ARRAY of STRING)
#
# Print the schema
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.types import *

medical_schema = StructType([
    StructField("record_id", IntegerType(), False),
    StructField("patient_id", IntegerType(), False),
    StructField("visit_date", DateType(), True),
    StructField("vitals", StructType([
        StructField("blood_pressure", StringType(), True),
        StructField("heart_rate", IntegerType(), True),
        StructField("temperature", DecimalType(4, 1), True)
    ]), True),
    StructField("notes", StringType(), True),
    StructField("medications", ArrayType(StringType()), True)
])

print("Medical Records Schema:")
for field in medical_schema.fields:
    print(f"  {field.name}: {field.dataType}")

---
# 📗 Section 4: Core Transformations

## 4.1 select() and selectExpr()

In [ ]:
from pyspark.sql.functions import *

# select() — choose columns
df = spark.table("techmart_dw.orders")
df.select("order_id", "customer_id", "total_amount", "status").show(5)

# selectExpr() — SQL expressions in select
df.selectExpr(
    "order_id",
    "total_amount",
    "total_amount * 0.08 AS tax",
    "CASE WHEN total_amount > 3000 THEN 'High' ELSE 'Low' END AS tier"
).show(5)

## 4.2 filter() / where()

In [ ]:
# filter with string expression
df.filter("status = 'completed' AND total_amount > 2000").show(5)

# filter with Column objects (more Pythonic, IDE-friendly)
df.filter(
    (col("status") == "completed") & (col("total_amount") > 2000)
).select("order_id", "status", "total_amount").show(5)

# Healthcare: filter claims
claims = spark.table("healthcare_dw.claims")
claims.filter(
    (col("status") == "denied") & (col("billed_amount") > 5000)
).select("claim_id", "patient_id", "billed_amount", "status").show(5)

## 4.3 withColumn() — Add/Modify Columns

In [ ]:
# Add new columns
orders = spark.table("techmart_dw.orders")
enriched = (orders
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("net_amount", col("total_amount") - col("discount_amount"))
    .withColumn("is_high_value", when(col("total_amount") > 3000, True).otherwise(False))
)
enriched.select("order_id", "year", "month", "net_amount", "is_high_value").show(5)

## 4.4-4.6 Rename, Drop, Deduplicate

In [ ]:
# withColumnRenamed
df = spark.table("techmart_dw.customers")
renamed = df.withColumnRenamed("first_name", "fname").withColumnRenamed("last_name", "lname")
print(f"Renamed columns: {renamed.columns[:5]}")

# drop columns
slim = df.drop("metadata", "created_at", "updated_at")
print(f"After drop: {len(slim.columns)} columns (was {len(df.columns)})")

# dropDuplicates
print(f"\nCustomers total: {df.count()}")
deduped = df.dropDuplicates(["email"])
print(f"After dedup on email: {deduped.count()}")

## 4.7-4.10 Sort, Limit, Union, Sample

In [ ]:
# orderBy / sort
orders = spark.table("techmart_dw.orders")
orders.orderBy(col("total_amount").desc()).select("order_id", "total_amount").show(5)

# union and unionByName
completed = orders.filter("status = 'completed'").select("order_id", "total_amount")
pending = orders.filter("status = 'pending'").select("order_id", "total_amount")
combined = completed.union(pending)
print(f"Combined: {combined.count():,} rows")

# sample
sample_df = orders.sample(fraction=0.01, seed=42)
print(f"1% sample: {sample_df.count()} rows")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Transform Healthcare Claims
# ============================================
# Using the healthcare_dw.claims table:
# 1. Filter to only 'approved' or 'paid' claims
# 2. Add column 'payment_ratio' = paid_amount / billed_amount
# 3. Add column 'claim_year' = year of claim_date
# 4. Drop the service_date column
# 5. Sort by billed_amount descending
# 6. Show top 10
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
claims = spark.table("healthcare_dw.claims")
result = (claims
    .filter(col("status").isin("approved", "paid"))
    .withColumn("payment_ratio", round(col("paid_amount") / col("billed_amount"), 3))
    .withColumn("claim_year", year("claim_date"))
    .drop("service_date")
    .orderBy(col("billed_amount").desc())
)
result.select("claim_id", "billed_amount", "paid_amount", "payment_ratio", "claim_year").show(10)

## 🎯 Practice: DataFrame Transformations

In [ ]:
# ============================================
# 🎯 YOUR TURN: Transform the Orders DataFrame
# ============================================
# Starting with the orders table, create a clean DataFrame that:
# 1. Filters to only 'completed' and 'shipped' orders
# 2. Adds a column 'order_year' extracted from order_date
# 3. Adds a column 'is_high_value' (True if total_amount > 500)
# 4. Renames 'total_amount' to 'revenue'
# 5. Drops the 'notes' and 'shipping_address' columns
# 6. Orders by revenue descending
# 7. Shows top 10
#
# Write your transformation chain below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, year, when, lit

clean_orders = (
    spark.table("techmart_dw.orders")
    # 1. Filter
    .filter(col("status").isin("completed", "shipped"))
    # 2. Add order_year
    .withColumn("order_year", year(col("order_date")))
    # 3. Add is_high_value
    .withColumn("is_high_value", when(col("total_amount") > 500, True).otherwise(False))
    # 4. Rename
    .withColumnRenamed("total_amount", "revenue")
    # 5. Drop columns
    .drop("notes", "shipping_address")
    # 6. Order
    .orderBy(col("revenue").desc())
)

# 7. Show
print("📊 Clean Orders (top 10 by revenue):")
clean_orders.show(10)
print(f"Total clean orders: {clean_orders.count():,}")


## ⚠️ Common PySpark Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| `df.collect()` on large data | OOM on driver | Use `.show()`, `.take()`, or write to table |
| Chaining `.count()` in loops | Triggers full recomputation each time | Cache first: `df.cache()` |
| Python UDFs for simple logic | 10-100x slower than built-in functions | Use `pyspark.sql.functions` |
| `df.filter(df.col == value)` | Creates new DF each time (not chained) | Chain: `df.filter(...).filter(...)` |
| Ignoring `explain()` | Can't see if plan is efficient | Always check plan for complex queries |
| Too many small files on write | Slow reads downstream | Use `coalesce()` before write |

---
# 📗 Section 5: Column Operations

## col(), lit(), when()/otherwise()

In [ ]:
from pyspark.sql.functions import *

# col() references a column, lit() creates a literal value
df = spark.table("techmart_dw.orders")
df.select(
    col("order_id"),
    col("total_amount"),
    lit("USD").alias("currency"),
    lit(0.08).alias("tax_rate"),
    (col("total_amount") * lit(0.08)).alias("tax_amount")
).show(5)

In [ ]:
# when/otherwise — conditional logic (like CASE WHEN)
orders = spark.table("techmart_dw.orders")
categorized = orders.withColumn("order_tier",
    when(col("total_amount") >= 4000, "Platinum")
    .when(col("total_amount") >= 2000, "Gold")
    .when(col("total_amount") >= 1000, "Silver")
    .otherwise("Bronze")
)
categorized.groupBy("order_tier").count().orderBy("count", ascending=False).show()

## String Functions

In [ ]:
# String operations on customer data
customers = spark.table("techmart_dw.customers")
cleaned = (customers
    .withColumn("first_name_clean", initcap(trim(col("first_name"))))
    .withColumn("email_domain", split(col("email"), "@").getItem(1))
    .withColumn("name_length", length(col("first_name")))
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
)
cleaned.select("first_name", "first_name_clean", "email_domain", "full_name").show(10, truncate=False)

## Date Functions

In [ ]:
# Date operations
orders = spark.table("techmart_dw.orders")
dated = (orders
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    .withColumn("order_dow", dayofweek("order_date"))
    .withColumn("days_ago", datediff(current_date(), col("order_date")))
    .withColumn("formatted", date_format("order_date", "MMM dd, yyyy"))
)
dated.select("order_id", "order_date", "order_year", "order_month", "days_ago", "formatted").show(5)

## NULL Handling

In [ ]:
# NULL operations
customers = spark.table("techmart_dw.customers")

# Count nulls per column
print("NULL counts:")
for c in ["email", "phone"]:
    null_count = customers.filter(col(c).isNull()).count()
    print(f"  {c}: {null_count} nulls")

# coalesce — first non-null value
filled = customers.withColumn("contact", coalesce(col("email"), col("phone"), lit("NO_CONTACT")))
filled.select("customer_id", "email", "phone", "contact").filter("email IS NULL").show(5)

# na.fill — fill nulls with defaults
filled2 = customers.na.fill({"email": "unknown@na.com", "phone": "000-000-0000"})
filled2.select("customer_id", "email", "phone").filter("customer_id % 10 = 0").show(5)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Clean Healthcare Patient Data
# ============================================
# Using healthcare_dw.patients:
# 1. Clean first_name: trim + initcap
# 2. Calculate age from dob (use datediff / 365.25)
# 3. Add age_group: <18='Pediatric', 18-64='Adult', 65+='Senior'
# 4. Handle any NULL dob with a default age_group of 'Unknown'
# 5. Show 10 rows with patient_id, first_name, dob, age, age_group
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
patients = spark.table("healthcare_dw.patients")
result = (patients
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("age", (datediff(current_date(), col("dob")) / 365.25).cast("int"))
    .withColumn("age_group",
        when(col("dob").isNull(), "Unknown")
        .when(col("age") < 18, "Pediatric")
        .when(col("age") <= 64, "Adult")
        .otherwise("Senior")
    )
)
result.select("patient_id", "first_name", "dob", "age", "age_group").show(10)

---
# 📗 Section 6: Aggregations

## groupBy() + agg()

In [ ]:
from pyspark.sql.functions import *

# Multiple aggregations in one call
orders = spark.table("techmart_dw.orders")
summary = (orders
    .groupBy("status")
    .agg(
        count("*").alias("order_count"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value"),
        min("total_amount").alias("min_order"),
        max("total_amount").alias("max_order"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy("total_revenue", ascending=False)
)
summary.show()

In [ ]:
# Healthcare: claims summary by provider specialty
claims = spark.table("healthcare_dw.claims")
providers = spark.table("healthcare_dw.providers")

specialty_summary = (claims
    .join(providers, "provider_id")
    .groupBy("specialty")
    .agg(
        count("*").alias("claim_count"),
        sum("billed_amount").alias("total_billed"),
        avg("paid_amount").alias("avg_paid"),
        countDistinct("patient_id").alias("unique_patients")
    )
    .orderBy("total_billed", ascending=False)
)
specialty_summary.show()

In [ ]:
# Pivot table: orders by status and payment method
pivot_df = (orders
    .groupBy("payment_method")
    .pivot("status", ["completed", "cancelled", "pending"])
    .agg(count("*"))
    .fillna(0)
)
pivot_df.show()

In [ ]:
# ============================================
# 🎯 YOUR TURN: Claims Summary
# ============================================
# Build a claims summary by diagnosis category:
# 1. Join claims with diagnoses on diagnosis_code = code
# 2. Group by category
# 3. Calculate: total_claims, total_billed, avg_billed, denial_rate
#    (denial_rate = denied claims / total claims)
# 4. Order by total_billed descending
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
claims = spark.table("healthcare_dw.claims")
diagnoses = spark.table("healthcare_dw.diagnoses")

result = (claims
    .join(diagnoses, claims.diagnosis_code == diagnoses.code, "inner")
    .groupBy("category")
    .agg(
        count("*").alias("total_claims"),
        round(sum("billed_amount"), 2).alias("total_billed"),
        round(avg("billed_amount"), 2).alias("avg_billed"),
        round(sum(when(col("status") == "denied", 1).otherwise(0)) / count("*") * 100, 2).alias("denial_rate_pct")
    )
    .orderBy("total_billed", ascending=False)
)
result.show()

In [ ]:
# Advanced aggregation patterns for DE
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, min as spark_min, max as spark_max,
    countDistinct, collect_list, collect_set, first, last,
    percentile_approx, stddev, variance
)

orders = spark.table("techmart_dw.orders")

# Pattern 1: Multiple aggregations in one pass
print("📊 ADVANCED AGGREGATION PATTERNS")
print("=" * 60)

summary = orders.agg(
    count("*").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    spark_sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_order_value"),
    percentile_approx("total_amount", 0.5).alias("median_order_value"),
    stddev("total_amount").alias("stddev_order_value"),
    spark_min("order_date").alias("first_order_date"),
    spark_max("order_date").alias("last_order_date"),
)
print("\n1️⃣ Full dataset summary:")
summary.show(truncate=False)

# Pattern 2: Conditional aggregation (CASE WHEN equivalent)
from pyspark.sql.functions import when

status_breakdown = orders.agg(
    count(when(col("status") == "completed", 1)).alias("completed"),
    count(when(col("status") == "shipped", 1)).alias("shipped"),
    count(when(col("status") == "pending", 1)).alias("pending"),
    count(when(col("status") == "cancelled", 1)).alias("cancelled"),
    spark_sum(when(col("status") == "completed", col("total_amount"))).alias("completed_revenue"),
)
print("\n2️⃣ Conditional aggregation (pivot-style):")
status_breakdown.show(truncate=False)

# Pattern 3: Collect values into arrays
from pyspark.sql.functions import collect_set, size

customer_methods = (
    orders
    .groupBy("customer_id")
    .agg(
        collect_set("payment_method").alias("payment_methods_used"),
        count("*").alias("order_count"),
    )
    .withColumn("num_methods", size("payment_methods_used"))
    .orderBy(col("num_methods").desc())
)
print("\n3️⃣ Collect into arrays (payment methods per customer):")
customer_methods.show(5, truncate=False)


In [ ]:
# ============================================
# 🎯 YOUR TURN: Aggregation Challenge
# ============================================
# Build a "daily business metrics" DataFrame with:
# - order_date
# - total_orders (count)
# - unique_customers (count distinct)
# - total_revenue (sum of total_amount)
# - avg_order_value
# - completed_pct (% of orders that are 'completed')
# - top_payment_method (most common payment method that day)
#
# Hint: For top_payment_method, you may need a window function or subquery
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, avg, round as spark_round, when
from pyspark.sql.window import Window

orders = spark.table("techmart_dw.orders")

daily_metrics = (
    orders
    .groupBy("order_date")
    .agg(
        count("*").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_order_value"),
        spark_round(
            count(when(col("status") == "completed", 1)) * 100.0 / count("*"), 1
        ).alias("completed_pct"),
    )
    .orderBy(col("order_date").desc())
)

print("📊 Daily Business Metrics:")
daily_metrics.show(10)


---
# 📗 Section 7: Joins (Comprehensive)

## All Join Types

In [ ]:
# Inner join
orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")

inner = orders.join(customers, "customer_id", "inner")
print(f"Inner join: {inner.count():,} rows")

# Left join — keep all orders even without matching customer
left = orders.join(customers, "customer_id", "left")
print(f"Left join: {left.count():,} rows")

# Anti join — orders with NO matching customer
anti = orders.join(customers, "customer_id", "left_anti")
print(f"Anti join (orphan orders): {anti.count():,} rows")

# Semi join — orders that HAVE a matching customer (but don't bring customer cols)
semi = orders.join(customers, "customer_id", "left_semi")
print(f"Semi join: {semi.count():,} rows")

In [ ]:
# Healthcare: complete claim view (3-table join)
claims = spark.table("healthcare_dw.claims")
patients = spark.table("healthcare_dw.patients")
providers = spark.table("healthcare_dw.providers")

complete_claims = (claims
    .join(patients, "patient_id", "left")
    .join(providers, "provider_id", "left")
    .select(
        "claim_id", "claim_date",
        concat_ws(" ", patients.first_name, patients.last_name).alias("patient_name"),
        patients.gender,
        providers.name.alias("provider_name"),
        providers.specialty,
        "billed_amount", "paid_amount", "status"
    )
)
complete_claims.show(10, truncate=False)

In [ ]:
# Handling duplicate column names
# Problem: both tables have "first_name"
df1 = spark.table("techmart_dw.customers").select("customer_id", "first_name")
df2 = spark.table("techmart_dw.employees").select(
    col("employee_id").alias("customer_id"),  # fake join key for demo
    col("first_name").alias("emp_first_name")
)

# Solution: rename before join or use aliases
joined = df1.join(df2, "customer_id", "inner")
print(f"Joined columns: {joined.columns}")
joined.show(5)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Complete Healthcare View
# ============================================
# Join claims + patients + providers + diagnoses to create:
# - patient_name, gender, age
# - provider_name, specialty
# - diagnosis category, is_chronic
# - billed_amount, paid_amount, status
# Filter to only chronic conditions with billed > 3000
# Show top 10 by billed_amount
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
claims = spark.table("healthcare_dw.claims")
patients = spark.table("healthcare_dw.patients")
providers = spark.table("healthcare_dw.providers")
diagnoses = spark.table("healthcare_dw.diagnoses")

result = (claims
    .join(patients, "patient_id")
    .join(providers, "provider_id")
    .join(diagnoses, claims.diagnosis_code == diagnoses.code, "left")
    .select(
        concat_ws(" ", patients.first_name, patients.last_name).alias("patient_name"),
        patients.gender,
        (datediff(current_date(), patients.dob) / 365.25).cast("int").alias("age"),
        providers.name.alias("provider_name"),
        providers.specialty,
        diagnoses.category,
        diagnoses.is_chronic,
        "billed_amount", "paid_amount", "status"
    )
    .filter((col("is_chronic") == True) & (col("billed_amount") > 3000))
    .orderBy(col("billed_amount").desc())
)
result.show(10, truncate=False)

## 🎯 Practice: Join Strategies & Optimization

In [ ]:
# ============================================
# 🎯 YOUR TURN: Optimize a Multi-Table Join
# ============================================
# Build a report that joins:
# - orders (20K rows)
# - customers (5K rows) 
# - order_items (50K rows)
# - products (500 rows)
#
# Requirements:
# 1. Only completed orders from 2023+
# 2. Show: customer_name, product_name, quantity, line_total
# 3. Use broadcast() for small tables
# 4. Filter BEFORE joining
# 5. Select only needed columns
#
# Write your optimized join below:


In [ ]:
# ============================================
# ✅ SOLUTION: Optimized Multi-Table Join
# ============================================
from pyspark.sql.functions import col, concat_ws, broadcast

# Step 1: Filter and select from large table FIRST
orders_filtered = (
    spark.table("techmart_dw.orders")
    .filter(col("status") == "completed")
    .filter(col("order_date") >= "2023-01-01")
    .select("order_id", "customer_id")  # Only needed columns!
)

# Step 2: Broadcast small tables
customers_small = broadcast(
    spark.table("techmart_dw.customers")
    .select("customer_id", 
            concat_ws(" ", col("first_name"), col("last_name")).alias("customer_name"))
)

products_small = broadcast(
    spark.table("techmart_dw.products")
    .select("product_id", "product_name", "category")
)

# Step 3: Join in optimal order (filtered large → small → medium)
result = (
    orders_filtered
    .join(customers_small, "customer_id")           # Broadcast (no shuffle)
    .join(spark.table("techmart_dw.order_items")
          .select("order_id", "product_id", "quantity", "line_total"),
          "order_id")                                # Shuffle (order_items is large)
    .join(products_small, "product_id")             # Broadcast (no shuffle)
    .select("customer_name", "product_name", "category", "quantity", "line_total")
    .orderBy(col("line_total").desc())
)

print("📊 Optimized Multi-Table Join Result:")
result.show(10)
print(f"Total rows: {result.count():,}")

print("\n✅ Optimizations applied:")
print("   1. Filtered orders BEFORE joining (20K → fewer rows)")
print("   2. Selected only needed columns (less data to shuffle)")
print("   3. Broadcast customers (5K rows) and products (500 rows)")
print("   4. Only order_items causes a shuffle (unavoidable, it is large)")


---
# 📗 Section 8: Window Functions

**WHY:** Window functions in PySpark mirror SQL windows but with a more composable API.
Define the window spec once, reuse across multiple calculations.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Define window specifications
orders = spark.table("techmart_dw.orders")

# Window: per customer, ordered by date
customer_window = Window.partitionBy("customer_id").orderBy("order_date")

# Window: per customer, all rows (for aggregates)
customer_all = Window.partitionBy("customer_id")

# Add window calculations
windowed = (orders
    .filter("status = 'completed'")
    .withColumn("row_num", row_number().over(customer_window))
    .withColumn("running_total", sum("total_amount").over(customer_window))
    .withColumn("customer_avg", avg("total_amount").over(customer_all))
    .withColumn("prev_order_amount", lag("total_amount", 1).over(customer_window))
    .withColumn("order_rank", dense_rank().over(
        Window.partitionBy("customer_id").orderBy(col("total_amount").desc())
    ))
)
windowed.filter("customer_id = 1").select(
    "order_id", "order_date", "total_amount", "row_num",
    "running_total", "customer_avg", "prev_order_amount", "order_rank"
).show(10)

In [ ]:
# Healthcare: running claim totals per patient
claims = spark.table("healthcare_dw.claims")

patient_window = Window.partitionBy("patient_id").orderBy("claim_date")
patient_all = Window.partitionBy("patient_id")

patient_claims = (claims
    .filter("status IN ('approved', 'paid')")
    .withColumn("claim_num", row_number().over(patient_window))
    .withColumn("running_billed", sum("billed_amount").over(patient_window))
    .withColumn("total_claims", count("*").over(patient_all))
    .withColumn("avg_claim", avg("billed_amount").over(patient_all))
    .withColumn("days_since_prev", datediff(
        col("claim_date"), lag("claim_date", 1).over(patient_window)
    ))
)
patient_claims.filter("patient_id = 1").select(
    "claim_id", "claim_date", "billed_amount", "claim_num",
    "running_billed", "days_since_prev"
).show(10)

In [ ]:
# Moving average (7-day window)
daily_orders = (spark.table("techmart_dw.orders")
    .filter("status = 'completed'")
    .groupBy("order_date")
    .agg(sum("total_amount").alias("daily_revenue"), count("*").alias("order_count"))
)

# Range-based window for 7-day moving average
day_window = Window.orderBy("order_date").rowsBetween(-6, 0)

with_ma = (daily_orders
    .withColumn("ma_7d", round(avg("daily_revenue").over(day_window), 2))
    .withColumn("running_total", sum("daily_revenue").over(Window.orderBy("order_date")))
    .orderBy("order_date")
)
with_ma.filter("order_date BETWEEN '2023-06-01' AND '2023-06-30'").show(15)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Patient Claim Windows
# ============================================
# For each patient, calculate:
# 1. Total number of claims
# 2. Running sum of billed_amount (ordered by claim_date)
# 3. Rank by billed_amount (highest = rank 1)
# 4. Difference from previous claim amount (lag)
# 5. Percentile rank (percent_rank)
# Filter to patient_id <= 5, show results
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.window import Window

claims = spark.table("healthcare_dw.claims")
w_ordered = Window.partitionBy("patient_id").orderBy("claim_date")
w_amount = Window.partitionBy("patient_id").orderBy(col("billed_amount").desc())
w_all = Window.partitionBy("patient_id")

result = (claims
    .filter("patient_id <= 5")
    .withColumn("total_claims", count("*").over(w_all))
    .withColumn("running_billed", sum("billed_amount").over(w_ordered))
    .withColumn("amount_rank", rank().over(w_amount))
    .withColumn("prev_amount", lag("billed_amount", 1).over(w_ordered))
    .withColumn("diff_from_prev", col("billed_amount") - col("prev_amount"))
    .withColumn("pct_rank", round(percent_rank().over(w_amount), 3))
)
result.select("patient_id", "claim_date", "billed_amount", "total_claims",
    "running_billed", "amount_rank", "diff_from_prev", "pct_rank").show(20)

In [ ]:
# Window function patterns for DE
from pyspark.sql.functions import col, row_number, rank, dense_rank, lag, lead, sum as spark_sum, avg
from pyspark.sql.window import Window

orders = spark.table("techmart_dw.orders")

# Pattern 1: Deduplication (most common DE use of window functions)
print("📊 WINDOW FUNCTION PATTERNS")
print("=" * 60)

# Deduplicate: keep only the LATEST order per customer
window_dedup = Window.partitionBy("customer_id").orderBy(col("order_date").desc())

deduped = (
    orders
    .withColumn("rn", row_number().over(window_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)
print(f"1️⃣ Deduplication: {orders.count():,} → {deduped.count():,} (latest per customer)")

# Pattern 2: Running totals
window_running = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_totals = (
    orders
    .filter(col("customer_id").isin(1, 2, 3))
    .withColumn("running_total", spark_sum("total_amount").over(window_running))
    .withColumn("order_number", row_number().over(Window.partitionBy("customer_id").orderBy("order_date")))
    .select("customer_id", "order_date", "total_amount", "running_total", "order_number")
    .orderBy("customer_id", "order_date")
)
print("\n2️⃣ Running totals per customer:")
running_totals.show(15)

# Pattern 3: Period-over-period comparison (LAG)
window_lag = Window.partitionBy("customer_id").orderBy("order_date")

with_prev = (
    orders
    .filter(col("customer_id") == 1)
    .withColumn("prev_amount", lag("total_amount", 1).over(window_lag))
    .withColumn("change", col("total_amount") - col("prev_amount"))
    .withColumn("change_pct", 
        when(col("prev_amount").isNotNull(), 
             ((col("total_amount") - col("prev_amount")) / col("prev_amount") * 100))
    )
    .select("order_date", "total_amount", "prev_amount", "change", "change_pct")
    .orderBy("order_date")
)
print("\n3️⃣ Period-over-period (customer 1):")
with_prev.show(10)


---
# 📗 Section 9: Complex Data Types

## JSON Parsing

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Parse JSON contact info from patients
patients = spark.table("healthcare_dw.patients")

# Define schema for the JSON
contact_schema = StructType([
    StructField("phone", StringType()),
    StructField("email", StringType()),
    StructField("emergency_contact", StringType())
])

# Parse JSON into struct columns
parsed = (patients
    .withColumn("contact", from_json(col("contact_json"), contact_schema))
    .withColumn("phone", col("contact.phone"))
    .withColumn("email", col("contact.email"))
    .withColumn("emergency", col("contact.emergency_contact"))
)
parsed.select("patient_id", "first_name", "phone", "email", "emergency").show(5, truncate=False)

In [ ]:
# get_json_object — extract single field without full parse
patients = spark.table("healthcare_dw.patients")
patients.select(
    "patient_id",
    get_json_object("contact_json", "$.phone").alias("phone"),
    get_json_object("contact_json", "$.email").alias("email")
).show(5, truncate=False)

## Arrays and Explode

In [ ]:
# Create array and explode
orders = spark.table("techmart_dw.orders")

# Create an array column
with_tags = orders.withColumn("tags",
    array(
        col("status"),
        col("payment_method"),
        col("order_source")
    )
)
with_tags.select("order_id", "tags").show(5, truncate=False)

# Explode: one row per array element
exploded = with_tags.select("order_id", explode("tags").alias("tag"))
exploded.show(10)
print(f"Original rows: {orders.count()}, After explode: {exploded.count()}")

In [ ]:
# collect_list / collect_set — aggregate into arrays
orders = spark.table("techmart_dw.orders")
customer_orders = (orders
    .filter("customer_id <= 5")
    .groupBy("customer_id")
    .agg(
        collect_list("status").alias("all_statuses"),
        collect_set("payment_method").alias("unique_payments"),
        count("*").alias("order_count")
    )
)
customer_orders.show(truncate=False)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Parse Patient Contact JSON
# ============================================
# 1. Read healthcare_dw.patients
# 2. Parse contact_json into separate columns (phone, email, emergency_contact)
# 3. Extract email domain (part after @)
# 4. Create a boolean 'has_emergency' column
# 5. Show 10 rows
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
patients = spark.table("healthcare_dw.patients")
contact_schema = StructType([
    StructField("phone", StringType()),
    StructField("email", StringType()),
    StructField("emergency_contact", StringType())
])

result = (patients
    .withColumn("contact", from_json("contact_json", contact_schema))
    .withColumn("phone", col("contact.phone"))
    .withColumn("email", col("contact.email"))
    .withColumn("emergency_contact", col("contact.emergency_contact"))
    .withColumn("email_domain", split(col("email"), "@").getItem(1))
    .withColumn("has_emergency", col("emergency_contact").isNotNull())
    .select("patient_id", "first_name", "phone", "email", "email_domain", "has_emergency")
)
result.show(10, truncate=False)

---
# 📗 Section 10: UDFs (User-Defined Functions)

**When to use:** Only when built-in functions can't do the job.
**Prefer:** Built-in functions > Pandas UDFs > Regular UDFs (performance order)

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, BooleanType
import re

# Regular UDF: validate diagnosis code format
@udf(returnType=BooleanType())
def is_valid_diagnosis(code):
    if code is None:
        return False
    # Valid format: one letter followed by 3 digits
    return bool(re.match(r'^[A-E]\d{3}$', code))

claims = spark.table("healthcare_dw.claims")
validated = claims.withColumn("is_valid_code", is_valid_diagnosis(col("diagnosis_code")))
validated.groupBy("is_valid_code").count().show()

In [ ]:
# Pandas UDF (vectorized — MUCH faster for large datasets)
import pandas as pd

@pandas_udf(StringType())
def categorize_amount(amounts: pd.Series) -> pd.Series:
    return amounts.apply(
        lambda x: "High" if x > 5000 else "Medium" if x > 2000 else "Low"
    )

claims = spark.table("healthcare_dw.claims")
result = claims.withColumn("amount_category", categorize_amount(col("billed_amount")))
result.groupBy("amount_category").agg(
    count("*").alias("count"),
    round(avg("billed_amount"), 2).alias("avg_amount")
).show()

---
# 📗 Section 11: Spark SQL Integration

## Mixing DataFrame API and SQL

In [ ]:
# Create temp view from DataFrame
orders = spark.table("techmart_dw.orders")
high_value = orders.filter("total_amount > 3000 AND status = 'completed'")
high_value.createOrReplaceTempView("high_value_orders")

# Now query with SQL
result = spark.sql("""
    SELECT payment_method, COUNT(*) AS cnt, ROUND(AVG(total_amount), 2) AS avg_amt
    FROM high_value_orders
    GROUP BY payment_method
    ORDER BY cnt DESC
""")
result.show()

In [ ]:
# Side-by-side: same result, two approaches
# SQL approach
sql_result = spark.sql("""
    SELECT status, COUNT(*) AS cnt, SUM(total_amount) AS revenue
    FROM techmart_dw.orders
    GROUP BY status ORDER BY revenue DESC
""")

# DataFrame approach
from pyspark.sql.functions import *
df_result = (spark.table("techmart_dw.orders")
    .groupBy("status")
    .agg(count("*").alias("cnt"), sum("total_amount").alias("revenue"))
    .orderBy(col("revenue").desc())
)

print("SQL result:")
sql_result.show()
print("DataFrame result:")
df_result.show()

---
# 📗 Section 12: Performance Optimization

## explain() — Reading Execution Plans

In [ ]:
# Read execution plan
orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")

query = (orders
    .join(customers, "customer_id")
    .filter("status = 'completed'")
    .groupBy("customer_segment")
    .agg(sum("total_amount").alias("revenue"))
)
query.explain(True)

## Caching & Broadcast Joins

In [ ]:
from pyspark.sql.functions import broadcast
import time

# Cache a frequently-used DataFrame
orders = spark.table("techmart_dw.orders").filter("status = 'completed'")
orders.cache()

# First query (populates cache)
start = time.time()
orders.count()
print(f"First count (cache fill): {time.time()-start:.2f}s")

# Second query (from cache)
start = time.time()
orders.count()
print(f"Second count (from cache): {time.time()-start:.2f}s")

orders.unpersist()
print("Cache cleared.")

In [ ]:
# Broadcast join — for small dimension tables
from pyspark.sql.functions import broadcast

orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")  # smaller table

# Without broadcast (shuffle join)
print("Regular join plan:")
orders.join(customers, "customer_id").explain()

# With broadcast (no shuffle — small table sent to all executors)
print("\nBroadcast join plan:")
orders.join(broadcast(customers), "customer_id").explain()

## Repartition vs Coalesce

In [ ]:
# Check current partitions
orders = spark.table("techmart_dw.orders")
print(f"Current partitions: {orders.rdd.getNumPartitions()}")

# Repartition (increases or decreases, causes shuffle)
repartitioned = orders.repartition(8)
print(f"After repartition(8): {repartitioned.rdd.getNumPartitions()}")

# Coalesce (only decreases, NO shuffle — preferred for reducing)
coalesced = orders.coalesce(2)
print(f"After coalesce(2): {coalesced.rdd.getNumPartitions()}")

## Performance Deep Dive: Shuffle, Skew & Optimization

### The Shuffle Problem

A **shuffle** redistributes data across the cluster. It's the most expensive operation:

```
BEFORE SHUFFLE (data distributed randomly):
Executor 1: [A1, B2, C1, A3]
Executor 2: [B1, A2, C2, B3]
Executor 3: [C3, A4, B4, C4]

AFTER SHUFFLE (grouped by key for GROUP BY):
Executor 1: [A1, A2, A3, A4]  ← All A's together
Executor 2: [B1, B2, B3, B4]  ← All B's together
Executor 3: [C1, C2, C3, C4]  ← All C's together

Cost: Network I/O + Disk I/O + Serialization
```

### Operations That Cause Shuffles

| Operation | Shuffle? | Why |
|-----------|----------|-----|
| `filter()` | ❌ No | Each partition filters independently |
| `select()` | ❌ No | Column selection, no data movement |
| `withColumn()` | ❌ No | Row-level computation |
| `groupBy().agg()` | ✅ Yes | Must group same keys together |
| `join()` | ✅ Yes* | Must co-locate matching keys |
| `orderBy()` | ✅ Yes | Must sort across all partitions |
| `distinct()` | ✅ Yes | Must compare across partitions |
| `repartition()` | ✅ Yes | Explicitly redistributes |
| `coalesce()` | ❌ No | Just combines partitions |

*Join can avoid shuffle with broadcast join (small table fits in memory)

In [ ]:
# Performance optimization techniques
from pyspark.sql.functions import col, broadcast

orders = spark.table("techmart_dw.orders")
products = spark.table("techmart_dw.products")

print("🚀 PERFORMANCE OPTIMIZATION TECHNIQUES")
print("=" * 60)

# Technique 1: Broadcast join (avoid shuffle for small tables)
print("\n1️⃣ BROADCAST JOIN (small table → all executors)")
print(f"   Products table size: {products.count()} rows (small!)")
print(f"   Orders table size: {orders.count():,} rows (large)")

# Without broadcast: both tables shuffled (expensive)
# With broadcast: products sent to all executors (cheap)
result = orders.join(broadcast(products), 
                     orders.product_id == products.product_id, 
                     "inner") if hasattr(orders, 'product_id') else None

print("   ✅ broadcast(products) sends small table to all executors")
print("   ✅ No shuffle needed for the large orders table")

# Technique 2: Filter early (reduce data before expensive operations)
print("\n2️⃣ FILTER EARLY (predicate pushdown)")
# Bad: join everything, then filter
# Good: filter first, then join
filtered_orders = orders.filter(col("status") == "completed")
print(f"   Before filter: {orders.count():,} rows")
print(f"   After filter: {filtered_orders.count():,} rows")
print("   ✅ Always filter BEFORE joins and aggregations")

# Technique 3: Select only needed columns (column pruning)
print("\n3️⃣ COLUMN PRUNING")
all_cols = len(orders.columns)
needed = orders.select("order_id", "customer_id", "total_amount", "order_date")
print(f"   All columns: {all_cols}")
print(f"   Selected: {len(needed.columns)}")
print("   ✅ Less data to read, shuffle, and process")

# Technique 4: Cache for repeated access
print("\n4️⃣ CACHING")
print("   df.cache() — keeps DataFrame in memory for reuse")
print("   df.unpersist() — removes from cache when done")
print("   ✅ Use when: same DataFrame accessed multiple times")
print("   ❌ Don't use when: DataFrame used only once")


In [ ]:
# ============================================
# 🎯 YOUR TURN: Optimize This Query
# ============================================
# The following query is SLOW. Identify the problems and fix them.
#
# SLOW VERSION:
# result = (
#     spark.table("techmart_dw.orders")                    # Read ALL columns
#     .join(spark.table("techmart_dw.customers"),          # Shuffle join
#           "customer_id")
#     .join(spark.table("techmart_dw.products"),           # Another shuffle
#           "product_id")  
#     .filter(col("status") == "completed")               # Filter AFTER joins!
#     .groupBy("customer_segment", "category")
#     .agg(sum("total_amount").alias("revenue"))
#     .orderBy("revenue")
# )
#
# Problems:
# 1. Reading all columns (should select only needed)
# 2. Filtering AFTER joins (should filter BEFORE)
# 3. Not broadcasting small tables
# 4. Missing desc() on orderBy
#
# Write the OPTIMIZED version below:


In [ ]:
# ============================================
# ✅ SOLUTION: Optimized Query
# ============================================
from pyspark.sql.functions import col, sum as spark_sum, broadcast

# Optimized: filter early, select needed columns, broadcast small tables
result = (
    spark.table("techmart_dw.orders")
    .select("order_id", "customer_id", "total_amount", "status")  # Column pruning
    .filter(col("status") == "completed")                          # Filter EARLY
    .join(
        broadcast(spark.table("techmart_dw.customers")             # Broadcast small table
            .select("customer_id", "customer_segment")),
        "customer_id"
    )
    .groupBy("customer_segment")
    .agg(spark_sum("total_amount").alias("revenue"))
    .orderBy(col("revenue").desc())                                # DESC for top results
)

print("📊 Optimized Result:")
result.show()

print("\n✅ Optimizations applied:")
print("   1. Selected only needed columns (less I/O)")
print("   2. Filtered BEFORE join (fewer rows to shuffle)")
print("   3. Broadcast small customers table (no shuffle)")
print("   4. Removed unnecessary products join")
print("   5. Added .desc() for meaningful ordering")


---
# 📗 Section 13: Writing Data

In [ ]:
# Write as Delta table (overwrite mode)
spark.sql("USE healthcare_dw")

claims = spark.table("claims")
silver_claims = (claims
    .filter(col("diagnosis_code").isNotNull())
    .withColumn("claim_year", year("claim_date"))
    .withColumn("claim_month", month("claim_date"))
    .dropDuplicates(["claim_id"])
)

# Write as managed table
silver_claims.write.format("delta").mode("overwrite").saveAsTable("silver_claims")
print(f"Written silver_claims: {spark.table('silver_claims').count():,} rows")

In [ ]:
# Write partitioned table
silver_claims = spark.table("silver_claims")
(silver_claims
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("claim_year")
    .saveAsTable("silver_claims_partitioned")
)
print("Written partitioned table by claim_year")
spark.sql("SHOW PARTITIONS silver_claims_partitioned").show()

---
# 🚀 Section 14: Mini Projects

## Project 1: Healthcare Data Pipeline (Bronze → Silver)

In [ ]:
# Bronze → Silver pipeline for claims
from pyspark.sql.functions import *

# Bronze: raw claims (already loaded)
bronze_claims = spark.table("healthcare_dw.claims")
print(f"Bronze: {bronze_claims.count():,} rows")

# Silver: clean, validate, deduplicate
silver = (bronze_claims
    # Deduplicate
    .dropDuplicates(["claim_id"])
    # Validate
    .filter(col("claim_id").isNotNull())
    .filter(col("patient_id").isNotNull())
    .filter(col("billed_amount") > 0)
    # Clean
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("claim_type", lower(trim(col("claim_type"))))
    # Enrich
    .withColumn("claim_year", year("claim_date"))
    .withColumn("claim_month", month("claim_date"))
    .withColumn("payment_ratio", round(col("paid_amount") / col("billed_amount"), 4))
    .withColumn("days_to_process", datediff(col("claim_date"), col("service_date")))
    # Audit
    .withColumn("_processed_at", current_timestamp())
    .withColumn("_row_hash", md5(concat_ws("|",
        col("claim_id").cast("string"),
        col("billed_amount").cast("string"),
        col("status")
    )))
)

silver.write.format("delta").mode("overwrite").saveAsTable("healthcare_dw.silver_claims_v2")
print(f"Silver: {silver.count():,} rows")
print(f"Removed: {bronze_claims.count() - silver.count():,} rows (dupes/invalid)")

## Project 2: Patient Risk Scoring

In [ ]:
# Patient risk scoring using window functions and aggregations
from pyspark.sql.window import Window

claims = spark.table("healthcare_dw.silver_claims_v2")
patients = spark.table("healthcare_dw.patients")
diagnoses = spark.table("healthcare_dw.diagnoses")

# Calculate risk factors per patient
patient_risk = (claims
    .join(diagnoses, claims.diagnosis_code == diagnoses.code, "left")
    .groupBy("patient_id")
    .agg(
        count("*").alias("total_claims"),
        countDistinct("provider_id").alias("unique_providers"),
        sum("billed_amount").alias("total_billed"),
        avg("billed_amount").alias("avg_claim_amount"),
        sum(when(col("is_chronic") == True, 1).otherwise(0)).alias("chronic_claims"),
        sum(when(col("status") == "denied", 1).otherwise(0)).alias("denied_claims"),
        countDistinct("diagnosis_code").alias("unique_diagnoses")
    )
    # Score: higher = more risk
    .withColumn("risk_score",
        (col("total_claims") * 2) +
        (col("chronic_claims") * 5) +
        (col("denied_claims") * 3) +
        (col("unique_providers") * 1) +
        when(col("total_billed") > 50000, 10).otherwise(0)
    )
    .withColumn("risk_tier",
        when(col("risk_score") >= 50, "Critical")
        .when(col("risk_score") >= 30, "High")
        .when(col("risk_score") >= 15, "Medium")
        .otherwise("Low")
    )
    .join(patients.select("patient_id", "first_name", "last_name", "gender"), "patient_id")
    .orderBy(col("risk_score").desc())
)

patient_risk.select("patient_id", "first_name", "last_name", "total_claims",
    "chronic_claims", "total_billed", "risk_score", "risk_tier").show(15)

# Summary
patient_risk.groupBy("risk_tier").agg(
    count("*").alias("patients"),
    round(avg("risk_score"), 1).alias("avg_score"),
    round(avg("total_billed"), 2).alias("avg_billed")
).orderBy("avg_score", ascending=False).show()

## Project 3: Provider Performance Dashboard

In [ ]:
# Provider performance gold table
claims = spark.table("healthcare_dw.silver_claims_v2")
providers = spark.table("healthcare_dw.providers")

provider_metrics = (claims
    .join(providers, "provider_id")
    .groupBy("provider_id", "name", "specialty", "hospital")
    .agg(
        count("*").alias("total_claims"),
        countDistinct("patient_id").alias("unique_patients"),
        round(sum("billed_amount"), 2).alias("total_billed"),
        round(avg("billed_amount"), 2).alias("avg_claim"),
        round(avg("payment_ratio"), 4).alias("avg_payment_ratio"),
        round(sum(when(col("status") == "denied", 1).otherwise(0)) / count("*") * 100, 2).alias("denial_rate_pct"),
        round(avg("days_to_process"), 1).alias("avg_processing_days")
    )
    .withColumn("efficiency_score",
        (lit(100) - col("denial_rate_pct")) * col("avg_payment_ratio")
    )
    .orderBy(col("total_billed").desc())
)

provider_metrics.write.format("delta").mode("overwrite").saveAsTable("healthcare_dw.gold_provider_metrics")
provider_metrics.show(10)

# Top specialties
provider_metrics.groupBy("specialty").agg(
    count("*").alias("providers"),
    round(sum("total_billed"), 2).alias("specialty_revenue"),
    round(avg("denial_rate_pct"), 2).alias("avg_denial_rate")
).orderBy("specialty_revenue", ascending=False).show()

---
# 🏆 Section 15: Interview Challenges

## Challenge 1: Deduplicate Claims (Keep Latest)

In [ ]:
from pyspark.sql.window import Window

claims = spark.table("healthcare_dw.claims")

# Deduplicate: keep latest claim per claim_id (by claim_date)
w = Window.partitionBy("claim_id").orderBy(col("claim_date").desc())
deduped = (claims
    .withColumn("rn", row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)
print(f"Original: {claims.count():,}")
print(f"Deduped: {deduped.count():,}")
print(f"Removed: {claims.count() - deduped.count():,} duplicates")

## Challenge 2: Patients with Claims in 3+ Consecutive Months

In [ ]:
# Consecutive months using gaps-and-islands
from pyspark.sql.functions import *
from pyspark.sql.window import Window

claims = spark.table("healthcare_dw.claims")

# Get distinct patient-month combinations
patient_months = (claims
    .withColumn("claim_ym", date_format("claim_date", "yyyy-MM"))
    .withColumn("month_num", year("claim_date") * 12 + month("claim_date"))
    .select("patient_id", "claim_ym", "month_num")
    .distinct()
)

# Gaps and islands
w = Window.partitionBy("patient_id").orderBy("month_num")
islands = (patient_months
    .withColumn("rn", row_number().over(w))
    .withColumn("island", col("month_num") - col("rn"))
)

# Find streaks of 3+
streaks = (islands
    .groupBy("patient_id", "island")
    .agg(
        count("*").alias("consecutive_months"),
        min("claim_ym").alias("streak_start"),
        max("claim_ym").alias("streak_end")
    )
    .filter("consecutive_months >= 3")
    .orderBy(col("consecutive_months").desc())
)
streaks.show(15)
print(f"Patients with 3+ consecutive months: {streaks.select('patient_id').distinct().count()}")

## Challenge 3: Month-over-Month Claim Growth

In [ ]:
# MoM growth by claim type
claims = spark.table("healthcare_dw.claims")

monthly = (claims
    .withColumn("month", date_format("claim_date", "yyyy-MM"))
    .groupBy("month", "claim_type")
    .agg(count("*").alias("claims"), sum("billed_amount").alias("revenue"))
)

w = Window.partitionBy("claim_type").orderBy("month")
growth = (monthly
    .withColumn("prev_revenue", lag("revenue").over(w))
    .withColumn("mom_growth_pct", 
        round((col("revenue") - col("prev_revenue")) / col("prev_revenue") * 100, 2))
    .filter("prev_revenue IS NOT NULL")
    .orderBy("claim_type", "month")
)
growth.show(20)

## Challenge 4: Potential Fraud Detection

In [ ]:
# Fraud: same patient, same day, multiple providers
claims = spark.table("healthcare_dw.claims")

suspicious = (claims
    .groupBy("patient_id", "claim_date")
    .agg(
        countDistinct("provider_id").alias("num_providers"),
        count("*").alias("num_claims"),
        sum("billed_amount").alias("total_billed")
    )
    .filter("num_providers >= 3")
    .orderBy(col("total_billed").desc())
)
print(f"Suspicious cases (3+ providers same day): {suspicious.count()}")
suspicious.show(10)

## Challenge 5: Patient Journey Timeline

In [ ]:
# Patient journey with window functions
from pyspark.sql.window import Window

claims = spark.table("healthcare_dw.claims")
providers = spark.table("healthcare_dw.providers")

w = Window.partitionBy("patient_id").orderBy("claim_date")

journey = (claims
    .join(providers, "provider_id")
    .filter("patient_id <= 3")
    .withColumn("visit_num", row_number().over(w))
    .withColumn("days_since_prev", datediff(col("claim_date"), lag("claim_date").over(w)))
    .withColumn("running_cost", sum("billed_amount").over(w))
    .withColumn("prev_specialty", lag("specialty").over(w))
    .withColumn("specialty_change", 
        when(col("specialty") != col("prev_specialty"), True).otherwise(False))
    .select("patient_id", "visit_num", "claim_date", "specialty",
        "billed_amount", "days_since_prev", "running_cost", "specialty_change")
    .orderBy("patient_id", "visit_num")
)
journey.show(30, truncate=False)

---
# ✅ Notebook Complete!

**What was covered:**
- Spark architecture: drivers, executors, DAGs, lazy evaluation, shuffles
- Reading data: spark.table(), spark.read, schema enforcement
- Schema definition: StructType, nested schemas, DDL strings
- Core transformations: select, filter, withColumn, join, sort, union
- Column operations: when/otherwise, string/date/null functions
- Aggregations: groupBy, agg, pivot, conditional aggregation
- Joins: inner, left, anti, semi, broadcast
- Window functions: row_number, lag/lead, running totals, moving averages
- Complex types: JSON parsing, arrays, explode, collect_list
- UDFs: regular and Pandas (vectorized)
- Performance: explain, cache, broadcast, repartition
- Writing: Delta tables, partitioned writes
- NEW: Healthcare Claims dataset (5 tables, 100K+ rows)
- 3 mini projects + 5 interview challenges

**Next:** Notebook 04 — Medallion Architecture

---
*Both `techmart_dw` and `healthcare_dw` databases remain available.*